# ML-06 — Signal audit and sensitivity

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w04_signal_audit.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Signal evidence

Held-out permutation sensitivity measures change in weighted error; it does not identify causal Google signals.

In [2]:
receipt_path=ROOT/'work/outputs/ctr_metrics.json'
if not receipt_path.exists():
    study.run()
r=json.loads(receipt_path.read_text())
print(pd.DataFrame(r['permutation_importance']).sort_values('weighted_mae_increase_pp',ascending=False).to_string(index=False))

               feature  weighted_mae_increase_pp    sd_pp
          avg_position              2.814934e-02 0.003991
      content_age_days              3.784323e-03 0.004266
        log_word_count              2.910358e-03 0.001399
     log_search_volume              1.581219e-03 0.000753
           main_intent              1.063994e-03 0.000217
           competition              4.980210e-04 0.000213
          content_type              4.439201e-07 0.000001
               log_cpc             -2.103270e-04 0.000078
days_since_last_update             -5.068331e-04 0.000600


## Volume floor sensitivity

Re-score increasingly high-volume subsets without selecting a new model from test results.

In [3]:
print(pd.DataFrame(r['volume_sensitivity']).to_string(index=False))

 minimum_impressions   mae_pp  weighted_mae_pp  rmse_pp  observed_ctr_pp  predicted_ctr_pp    n
                 500 0.172950         0.172615 0.274226         0.285027          0.264124 3913
                1000 0.168675         0.172352 0.263850         0.287610          0.264725 3145
                3000 0.174633         0.173654 0.286889         0.296190          0.267812 1758


## Verdict

Position has the clearest association in this model. Smaller changes, including age and depth, need cautious interpretation and ablation.

In [4]:
print('Freshness has no positive permutation improvement in this run; do not claim it is a validated driver.')

Freshness has no positive permutation improvement in this run; do not claim it is a validated driver.
